In [1]:
!pip install langchain==0.3.17 langchain-community==0.3.13 langchain-huggingface
!pip install sentence-transformers rank-bm25 chromadb pypdf faiss-cpu
!pip install ragas groq python-dotenv ddgs langchain-groq datasets


INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is still looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce ru

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

In [1]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder
from groq import Groq
from ddgs import DDGS
from datasets import Dataset
from ragas import evaluate as ragas_evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from langchain_groq import ChatGroq
import chromadb
import uuid
import os
from dotenv import load_dotenv

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_1885/859532182.py:11: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipykernel_1885/859532182.py:11: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections imp

In [26]:
from datasets import Dataset
from ragas import evaluate as ragas_evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import llm_factory
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from groq import Groq
import os

In [9]:
from google.colab import files
uploaded = files.upload()

Saving RBI_Annual_report.pdf to RBI_Annual_report.pdf


In [2]:
load_dotenv()

False

In [6]:
from google.colab import userdata
import os
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

In [3]:
def load_pdf(path):
    reader = PdfReader(path)
    raw_text = ''
    for page in reader.pages:
        text = page.extract_text()
        if text:
            raw_text += text + '\n\n'
    return raw_text

def chunk_text(text):
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    return splitter.split_text(text)

def build_faiss(chunks):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    faiss_store = FAISS.from_texts(chunks, embeddings)
    return faiss_store

def build_chroma(chunks):
    client = chromadb.PersistentClient(path='./chroma_db')
    collection = client.get_or_create_collection('rbi_rag')
    for chunk in chunks:
        collection.add(
            documents=[chunk],
            ids=[str(uuid.uuid4())]
        )
    return collection

In [4]:
def generate(query, context):
    client = Groq(api_key=os.environ.get('GROQ_API_KEY'))
    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': f'''You are an expert financial analyst assistant. Answer the user's question ONLY using the provided context from the RBI Annual Report.
Do not use any outside knowledge or information from your training data.
If the answer is not present in the context, reply exactly: "This information is not available in the provided document."
Be precise, concise and factual. Cite specific numbers and facts directly from the context.

Context: {context}'''},
            {'role': 'user', 'content': query}
        ]
    )
    return response.choices[0].message.content

In [7]:
def normal_rag(query, collection):
    results = collection.query(
        query_texts=[query],
        n_results=5
    )
    return results['documents'][0]

def hybrid_retriever(query, chunks, faiss):
    faiss_retriever = faiss.as_retriever(search_kwargs={'k': 5})
    faiss_docs = faiss_retriever.invoke(query)

    bm25_retriever = BM25Retriever.from_texts(chunks)
    bm25_retriever.k = 5
    bm25_docs = bm25_retriever.invoke(query)

    # manual combination
    all_docs = faiss_docs + bm25_docs
    seen = set()
    unique_docs = []
    for doc in all_docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique_docs.append(doc)

    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    pairs = [[query, doc.page_content] for doc in unique_docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, unique_docs), key=lambda x: x[0], reverse=True)
    top_docs = [doc.page_content for _, doc in ranked[:5]]
    return top_docs

def evaluate_chunk(query, chunk):
    client = Groq(api_key=os.environ.get('GROQ_API_KEY'))
    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': '''Given a query and a context chunk, return ONLY one word:
RELEVANT, IRRELEVANT, or AMBIGUOUS
depending on whether the retrieved chunk is relevant, irrelevant or ambiguous to the query.'''},
            {'role': 'user', 'content': f'Query: {query}\n\nChunk: {chunk}'}
        ]
    )
    return response.choices[0].message.content.strip()

def crag(query, chunks, faiss):
    docs = hybrid_retriever(query, chunks, faiss)
    relevant = []
    do_web_search = False

    for chunk in docs:
        result = evaluate_chunk(query, chunk)
        if 'RELEVANT' in result.upper():
            relevant.append(chunk)
        elif 'IRRELEVANT' in result.upper():
            do_web_search = True
        elif 'AMBIGUOUS' in result.upper():
            relevant.append(chunk)
            do_web_search = True

    if do_web_search:
        results = DDGS().text(query, max_results=3)
        for r in results:
            relevant.append(r['body'])

    return relevant


In [10]:
PDF_PATH = "/content/RBI_Annual_report.pdf"

print("Loading and indexing PDF...")
text = load_pdf(PDF_PATH)
chunks = chunk_text(text)
print(f"Created {len(chunks)} chunks")

faiss_index = build_faiss(chunks)
collection = build_chroma(chunks)
print("Indexing done.")

# Query
query = "What was the food inflation rate in October 2024 and how much did it decline by March 2025?"

print("\nRunning retrievers...")
normal_context = normal_rag(query, collection)
hybrid_context = hybrid_retriever(query, chunks, faiss_index)
crag_context = crag(query, chunks, faiss_index)

print("\nGenerating answers...")
normal_answer = generate(query, "\n".join(normal_context))
hybrid_answer = generate(query, "\n".join(hybrid_context))
crag_answer = generate(query, "\n".join(crag_context))

print("\n" + "="*60)
print("NORMAL RAG:", normal_answer)
print("="*60)
print("HYBRID RAG:", hybrid_answer)
print("="*60)
print("CRAG:", crag_answer)
print("="*60)

Loading and indexing PDF...
Created 1466 chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Generating answers...

NORMAL RAG: The food inflation rate in October 2024 was 9.7 per cent. By March 2025, it declined to 2.9 per cent, which is a decline of 6.8 percentage points.
HYBRID RAG: According to the context, the food inflation rate in October 2024 was 9.7 per cent. By March 2025, it declined to 2.9 per cent. Therefore, the decline was 9.7 - 2.9 = 6.8 percentage points.
CRAG: The food inflation rate in October 2024 was 9.7 per cent. It declined to 2.9 per cent by March 2025, which is a decline of 6.8 percentage points.


In [12]:
def run_pipeline(query):
    print(f"\nQuery: {query}\n")

    normal_context = normal_rag(query, collection)
    hybrid_context = hybrid_retriever(query, chunks, faiss_index)
    crag_context = crag(query, chunks, faiss_index)

    normal_answer = generate(query, "\n".join(normal_context))
    hybrid_answer = generate(query, "\n".join(hybrid_context))
    crag_answer = generate(query, "\n".join(crag_context))

    print("="*60)
    print("NORMAL RAG:", normal_answer)
    print("="*60)
    print("HYBRID RAG:", hybrid_answer)
    print("="*60)
    print("CRAG:", crag_answer)
    print("="*60)

    return normal_context, hybrid_context, crag_context, normal_answer, hybrid_answer, crag_answer

In [14]:
normal_context, hybrid_context, crag_context, normal_answer, hybrid_answer, crag_answer = run_pipeline("What were the key reasons for the downward revision of India's GDP growth projection for 2024-25 and what monetary policy actions did RBI take in response?")


Query: What were the key reasons for the downward revision of India's GDP growth projection for 2024-25 and what monetary policy actions did RBI take in response?



Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NORMAL RAG: This information is not available in the provided document.
HYBRID RAG: The key reason for the downward revision of India's GDP growth projection for 2024-25 was the lower-than-expected growth of 5.4 per cent in Q2:2024-25. Consequently, the real GDP growth projection for 2024-25 was revised downwards to 6.6 per cent. 

In response, the Monetary Policy Committee (MPC) decided to keep the policy repo rate unchanged at 6.50 per cent with a 5-1 majority. Additionally, the MPC changed the monetary policy stance from withdrawal of accommodation to neutral, providing it the flexibility to monitor the progress and outlook on disinflation and growth and act in accordance with the evolving situation.
CRAG: The key reason for the downward revision of India's GDP growth projection for 2024-25 was the lower-than-expected growth of 5.4 per cent in Q2:2024-25. Consequently, the real GDP growth projection for 2024-25 was revised downwards to 6.6 per cent. 

In response, the Monetary Polic

In [28]:
import json

def simple_evaluate(query, context, answer, ground_truth):
    client = Groq(api_key=os.environ.get('GROQ_API_KEY'))
    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': '''You are an evaluator. Score the answer on two criteria:
1. Faithfulness (0-1): Is the answer grounded in the context only?
2. Relevancy (0-1): Does the answer actually address the question?
Return ONLY a JSON like: {"faithfulness": 0.9, "relevancy": 0.8}'''},
            {'role': 'user', 'content': f'Query: {query}\nContext: {context}\nAnswer: {answer}\nGround Truth: {ground_truth}'}
        ]
    )
    return json.loads(response.choices[0].message.content)

# Ground truths
ground_truths = {
    "What were the key reasons for the downward revision of India's GDP growth projection for 2024-25 and what monetary policy actions did RBI take in response?":
        "The GDP growth was revised downwards to 6.6% due to lower than expected Q2 growth of 5.4%. RBI kept repo rate unchanged at 6.50% and changed stance from withdrawal of accommodation to neutral.",

    "What was the food inflation rate in October 2024 and how much did it decline by March 2025?":
        "Food inflation peaked at 9.7% in October 2024 and declined to 2.9% by March 2025.",

    "What was the growth rate of electronic goods imports and what was the trade balance for electronic goods in 2024-25?":
        "Electronic goods imports grew 12.4% y-o-y to US$ 98.7 billion. Trade balance for electronic goods widened to US$ 60.1 billion."
}

# Run evaluation
results = []

for query, ground_truth in ground_truths.items():
    print(f"\nEvaluating: {query[:60]}...")

    normal_context, hybrid_context, crag_context, normal_answer, hybrid_answer, crag_answer = run_pipeline(query)

    normal_scores = simple_evaluate(query, "\n".join(normal_context), normal_answer, ground_truth)
    hybrid_scores = simple_evaluate(query, "\n".join(hybrid_context), hybrid_answer, ground_truth)
    crag_scores   = simple_evaluate(query, "\n".join(crag_context),   crag_answer,   ground_truth)

    results.append({
        "query": query[:60],
        "normal_faithfulness":  normal_scores['faithfulness'],
        "normal_relevancy":     normal_scores['relevancy'],
        "hybrid_faithfulness":  hybrid_scores['faithfulness'],
        "hybrid_relevancy":     hybrid_scores['relevancy'],
        "crag_faithfulness":    crag_scores['faithfulness'],
        "crag_relevancy":       crag_scores['relevancy'],
    })

# Print comparison table
print("\n" + "="*100)
print(f"{'Query':<45} | {'Normal':^15} | {'Hybrid':^15} | {'CRAG':^15}")
print(f"{'':^45} | {'F':^7} {'R':^7} | {'F':^7} {'R':^7} | {'F':^7} {'R':^7}")
print("="*100)
for r in results:
    print(f"{r['query']:<45} | {r['normal_faithfulness']:^7.2f} {r['normal_relevancy']:^7.2f} | {r['hybrid_faithfulness']:^7.2f} {r['hybrid_relevancy']:^7.2f} | {r['crag_faithfulness']:^7.2f} {r['crag_relevancy']:^7.2f}")
print("="*100)


Evaluating: What were the key reasons for the downward revision of India...

Query: What were the key reasons for the downward revision of India's GDP growth projection for 2024-25 and what monetary policy actions did RBI take in response?



Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NORMAL RAG: The growth rate of electronic goods imports was 12.4% (y-o-y) in 2024-25, and the trade balance for electronic goods widened marginally to a deficit of US$ 60.1 billion in 2024-25.
HYBRID RAG: The growth rate of electronic goods imports was 12.4 per cent (y-o-y) in 2024-25, and the trade balance for electronic goods widened marginally to a deficit of US$ 60.1 billion.
CRAG: The growth rate of electronic goods imports was 12.4% (y-o-y) in 2024-25, and the trade balance for electronic goods widened marginally to a deficit of US$ 60.1 billion.

Query                                         |     Normal      |     Hybrid      |      CRAG      
                                              |    F       R    |    F       R    |    F       R   
What were the key reasons for the downward revision of India |  1.00    0.00   |  0.90    0.90   |  1.00    1.00  
What was the food inflation rate in October 2024 and how muc |  1.00    1.00   |  1.00    1.00   |  1.00    1.00  
What was t